# Dataset provider - tree of Gaussian hierarchies

This notebook owns dataset definitions and generation. It does not contain neural training, learning rules, or evaluation metrics. `Eval.ipynb` can load another provider notebook through `AUTORD_DATASET_NOTEBOOK` as long as that notebook implements the provider contract below.

A provider exports `DATASET_PROVIDER_NAME`, `DEV_CASES`, `PROMOTION_CASES`, `DEV_SEEDS`, `PROMOTION_SEEDS`, `case_name(case)`, and `load_dataset(case, seed)`. `load_dataset` returns a mapping with `name`, `x`, `labels_by_level`, `k_levels`, and `parent_maps`; `metadata` is optional. Levels are ordered from leaf/fine to top/coarse.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import numpy as np

In [ ]:
@dataclass(frozen=True)
class GaussianHierarchyCase:
    name: str
    d: int
    k_levels: tuple[int, ...]
    sigma_levels: tuple[float, ...]
    samples_per_leaf: int
    top_extent: float = 5.0


DATASET_PROVIDER_NAME = 'nested-gaussian'

DEV_CASES = [
    GaussianHierarchyCase('gaussian-small', 6, (18, 6, 3), (0.16, 0.50, 1.10), 60),
    GaussianHierarchyCase('gaussian-medium', 8, (24, 8, 3), (0.20, 0.65, 1.30), 50),
]
PROMOTION_CASES = DEV_CASES + [
    GaussianHierarchyCase('gaussian-large', 10, (30, 10, 4), (0.19, 0.60, 1.40), 45),
]
DEV_SEEDS = [3, 11, 29]
PROMOTION_SEEDS = [3, 11, 29, 47, 83]

In [ ]:
def _balanced_parent_map(n_child, n_parent, rng):
    ids = np.arange(n_child) % n_parent
    rng.shuffle(ids)
    return ids


def case_name(case):
    return case.name


def load_dataset(case: GaussianHierarchyCase, seed: int):
    if len(case.k_levels) < 1:
        raise ValueError('k_levels must contain at least one level')
    if len(case.sigma_levels) != len(case.k_levels):
        raise ValueError('sigma_levels must match k_levels')

    rng = np.random.default_rng(seed)
    level_count = len(case.k_levels)
    centers = [None] * level_count
    centers[-1] = rng.uniform(
        -case.top_extent, case.top_extent, size=(case.k_levels[-1], case.d)
    )
    parent_maps = [None] * (level_count - 1)

    for level in range(level_count - 2, -1, -1):
        parent_map = _balanced_parent_map(
            case.k_levels[level], case.k_levels[level + 1], rng
        )
        parent_maps[level] = parent_map
        centers[level] = centers[level + 1][parent_map] + rng.normal(
            0,
            case.sigma_levels[level + 1],
            size=(case.k_levels[level], case.d),
        )

    leaf_ids = np.repeat(np.arange(case.k_levels[0]), case.samples_per_leaf)
    x = centers[0][leaf_ids] + rng.normal(
        0, case.sigma_levels[0], size=(len(leaf_ids), case.d)
    )
    labels = [leaf_ids]
    current = leaf_ids
    for parent_map in parent_maps:
        current = parent_map[current]
        labels.append(current.copy())

    permutation = rng.permutation(len(x))
    return {
        'name': f'{case.name}/seed-{seed}',
        'x': x[permutation],
        'labels_by_level': tuple(label[permutation] for label in labels),
        'k_levels': tuple(case.k_levels),
        'parent_maps': tuple(parent_maps),
        'metadata': {
            'provider': DATASET_PROVIDER_NAME,
            'case': case.name,
            'data_seed': int(seed),
            'centers_by_level': tuple(centers),
        },
    }

## Visualize a generated hierarchy

The following interactive example uses `src/util/plot_util.py` to inspect the same high-dimensional Gaussian dataset in several complementary ways: a shared PCA projection colored at each hierarchy level, a PCA variance spectrum, a feature-correlation heatmap, and the ground-truth parent-child tree. Change `example_case` or `example_seed` to inspect another configured dataset.

The cell is tagged `skip-on-provider-import`, so `Eval.ipynb` does not execute plotting code when it loads this notebook as a provider.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

repo_candidates = (Path.cwd(), Path.cwd().parent)
repo_root = next(
    (candidate.resolve() for candidate in repo_candidates
     if (candidate / 'src' / 'util' / 'plot_util.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('run this notebook from the repository root or nb directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.util.plot_util import (
    plot_feature_correlation,
    plot_hierarchy_tree,
    plot_pca_hierarchy,
    plot_variance_spectrum,
)

example_case = DEV_CASES[0]
example_seed = DEV_SEEDS[0]
example = load_dataset(example_case, example_seed)
level_count = len(example['k_levels'])
level_names = [f'Level {level}' for level in range(level_count)]
if level_count == 1:
    level_names[0] = 'Only level'
else:
    level_names[0] = 'Leaf / fine'
    level_names[-1] = 'Top / coarse'

projection_figure, _ = plot_pca_hierarchy(
    example['x'],
    example['labels_by_level'],
    level_names=level_names,
    max_points=2500,
    seed=example_seed,
)
projection_figure.suptitle(
    f"{example['name']} - samples colored by hierarchy level", fontsize=14
)

diagnostic_figure, diagnostic_axes = plt.subplots(
    1, 2, figsize=(12, 4.5), constrained_layout=True
)
plot_variance_spectrum(example['x'], ax=diagnostic_axes[0])
plot_feature_correlation(example['x'], ax=diagnostic_axes[1])
diagnostic_figure.suptitle(f"{example['name']} - high-dimensional diagnostics")

tree_figure, _ = plot_hierarchy_tree(
    example['parent_maps'],
    example['k_levels'],
    level_names=level_names,
)
tree_figure.suptitle(f"{example['name']} - ground-truth hierarchy")
plt.show()

## Adding another dataset provider

Create a notebook that exports the same provider names. An open-source provider may ignore the data seed when loading a fixed split, but each returned dataset must include one integer label vector per hierarchy level and a child-to-parent map for every adjacent pair. Set `AUTORD_DATASET_NOTEBOOK` to that notebook's path before launching `Eval.ipynb`; no training or metric code needs to change.